# 🔎 Búsqueda masiva de cuentas en archivo plano (.txt)

Este cuaderno automatiza el proceso manual de:
1. Leer **miles de números de cuenta** desde un Excel (columna **D**, desde la fila **2**).
2. Buscarlos dentro de un **archivo plano `.txt`** de gran tamaño (millones de líneas).
3. Guardar cada **línea/registro** donde aparece una cuenta buscada.
4. Exportar el resultado a **Excel (.xlsx)** y **TXT (.txt)** dentro de Google Drive.

> **Cómo usarlo:** ejecuta los bloques **en orden, de arriba hacia abajo** (botón ▶️ de cada celda
> o `Shift + Enter`). Las partes que puedes ajustar están marcadas con `# 👉 PUEDES MODIFICAR`.

---
### ⚙️ Decisiones de diseño importantes (léelo una vez)
- **Coincidencia EXACTA por número:** la cuenta `123` **no** se confunde con `1234` ni con `99123`.
  Esto se logra extrayendo los *números completos* de cada línea y comparándolos con el conjunto de cuentas.
  Si tus cuentas tienen **letras o guiones**, cambia el patrón en el Bloque 5 (hay un comentario que lo explica).
- **Memoria:** el `.txt` se lee **línea por línea** (no se carga entero) y las cuentas se guardan en un
  `set` para que cada búsqueda sea instantánea. Apto para millones de líneas y miles de cuentas.


## 🧩 Bloque 1 — Instalación de dependencias
En Google Colab `pandas` ya viene instalado; aseguramos `openpyxl` para leer/escribir `.xlsx`.

In [ ]:
# Instala (o confirma) las librerías necesarias. La 'q' = modo silencioso.
!pip install openpyxl -q
print("✅ Dependencias listas.")

## 🔗 Bloque 2 — Conexión a Google Drive
Al ejecutar, Colab pedirá permiso para acceder a tu Drive. Acepta con tu cuenta.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive conectado. Tus archivos están bajo: /content/drive/MyDrive")

## 📂 Bloque 3 — Selección de archivos (navegador interactivo)
Aparecerán **dos selectores**. En cada uno:
- 📁 = carpeta → selecciónala y pulsa **«Entrar / Seleccionar»** para abrirla.
- `.. (subir un nivel)` → vuelve a la carpeta anterior.
- 📄 = archivo válido → selecciónalo y pulsa el botón; verás **✅ Archivo seleccionado**.

Selecciona **(1)** el Excel de cuentas y **(2)** el archivo plano `.txt`.

In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display

# 👉 PUEDES MODIFICAR: carpeta donde empieza el navegador
RUTA_INICIAL = '/content/drive/MyDrive'

def crear_selector(titulo, ruta_inicial, extensiones_validas):
    """Crea un navegador de carpetas/archivos. Devuelve un dict con la ruta elegida."""
    estado = {'ruta': ruta_inicial, 'archivo': None}

    etiqueta_ruta = widgets.HTML()
    etiqueta_sel  = widgets.HTML(value="<i>Aún no has seleccionado archivo.</i>")
    lista = widgets.Select(options=[], rows=12, layout=widgets.Layout(width='95%'))
    boton = widgets.Button(description='Entrar / Seleccionar', button_style='info',
                           layout=widgets.Layout(width='220px'))

    def listar():
        ruta = estado['ruta']
        etiqueta_ruta.value = (f"<b>{titulo}</b><br>📂 Carpeta actual: <code>{ruta}</code>")
        try:
            nombres = sorted(os.listdir(ruta))
        except Exception as e:
            nombres = []
            etiqueta_ruta.value += f"<br><span style='color:red'>No se pudo abrir: {e}</span>"
        entradas = [('.. (subir un nivel)', '__UP__')]
        # Primero las carpetas
        for n in nombres:
            full = os.path.join(ruta, n)
            if os.path.isdir(full):
                entradas.append((f'📁 {n}', full))
        # Luego los archivos con la extensión válida
        for n in nombres:
            full = os.path.join(ruta, n)
            if os.path.isfile(full) and n.lower().endswith(extensiones_validas):
                entradas.append((f'📄 {n}', full))
        lista.options = entradas

    def on_click(_):
        valor = lista.value
        if valor is None:
            return
        if valor == '__UP__':
            estado['ruta'] = os.path.dirname(estado['ruta'].rstrip('/')) or '/'
            listar()
        elif os.path.isdir(valor):
            estado['ruta'] = valor
            listar()
        elif os.path.isfile(valor):
            estado['archivo'] = valor
            etiqueta_sel.value = f"✅ <b>Archivo seleccionado:</b> <code>{valor}</code>"

    boton.on_click(on_click)
    listar()
    display(widgets.VBox([etiqueta_ruta, lista, boton, etiqueta_sel]))
    return estado

print(">>> 1) Selecciona el ARCHIVO EXCEL con las cuentas:")
sel_excel = crear_selector('1) EXCEL de cuentas', RUTA_INICIAL, ('.xlsx', '.xls'))

print("\n>>> 2) Selecciona el ARCHIVO PLANO (.txt):")
sel_txt = crear_selector('2) ARCHIVO PLANO (.txt)', RUTA_INICIAL, ('.txt', '.csv', '.dat'))

## 📊 Bloque 4 — Lectura del Excel (columna D, desde la fila 2)
- Toma la **columna D** (4.ª columna) a partir de la **fila 2**.
- Ignora filas vacías, quita espacios y trata todo como **texto** (para no perder ceros a la izquierda).

> Nota: para que los **ceros a la izquierda** sobrevivan, la columna en Excel debe estar como **texto**.
> Si Excel guardó la cuenta como número, el cero inicial ya no existe en el archivo original.

In [ ]:
import pandas as pd

# 👉 Si el navegador falló, puedes pegar la ruta a mano entre comillas:
RUTA_EXCEL = sel_excel['archivo']   # ej: '/content/drive/MyDrive/MiCarpeta/cuentas.xlsx'

if not RUTA_EXCEL:
    raise ValueError("No seleccionaste el Excel en el Bloque 3 (o pega la ruta manualmente).")

# header=None => leemos por POSICIÓN. dtype=str => todo como texto.
df_excel = pd.read_excel(RUTA_EXCEL, header=None, dtype=str)

# Columna D = índice 3 ; desde la fila 2 = índice 1 en adelante.
serie = df_excel.iloc[1:, 3]

# Limpieza: quitar vacíos, espacios y el ".0" que a veces deja Excel con números.
serie = serie.dropna().astype(str).str.strip()
serie = serie[serie != '']
serie = serie.str.replace(r'\.0$', '', regex=True)   # 123.0 -> 123

cuentas_lista = serie.tolist()       # todas las leídas (puede incluir repetidas)
cuentas_set   = set(cuentas_lista)   # únicas, para búsquedas O(1)

print(f"✅ Cuentas leídas (filas):   {len(cuentas_lista):,}")
print(f"✅ Cuentas únicas a buscar:  {len(cuentas_set):,}")
print("Ejemplos:", cuentas_lista[:5])

## 🗂️ Bloque 5 — Procesamiento del archivo plano (.txt)
Lee el `.txt` **línea por línea**. De cada línea extrae los **números completos** y los compara
con el conjunto de cuentas (coincidencia **exacta**). Guarda la línea por cada cuenta encontrada.

In [ ]:
import re

RUTA_TXT = sel_txt['archivo']   # 👉 o pega la ruta a mano: '/content/drive/MyDrive/.../archivo.txt'
if not RUTA_TXT:
    raise ValueError("No seleccionaste el .txt en el Bloque 3 (o pega la ruta manualmente).")

# 👉 PUEDES MODIFICAR:
ENCODING = 'utf-8'   # Si ves acentos raros (Ã±, Ã©), cambia a 'latin-1'.

# Patrón de "qué es una cuenta dentro de la línea":
#   r'\d+'        -> números puros (RECOMENDADO).  123 NO coincide con 1234.
#   r'[A-Za-z0-9]+' -> usa este si tus cuentas tienen letras.
patron = re.compile(r'\d+')

resultados = []              # lista de tuplas (cuenta, linea_completa)
cuentas_encontradas = set()  # cuentas distintas que sí aparecieron

print("⏳ Procesando el archivo plano... (puede tardar en archivos muy grandes)")
contador = 0
with open(RUTA_TXT, 'r', encoding=ENCODING, errors='replace') as f:
    for linea in f:
        contador += 1
        linea = linea.rstrip('\n').rstrip('\r')          # quitar salto de línea
        tokens = set(patron.findall(linea))              # números de esta línea
        comunes = tokens & cuentas_set                   # ¿alguno es una cuenta buscada?
        if comunes:
            for cuenta in comunes:
                resultados.append((cuenta, linea))       # una fila por (cuenta, línea)
                cuentas_encontradas.add(cuenta)
        if contador % 200000 == 0:                       # señal de avance
            print(f"   ... {contador:,} líneas leídas | {len(resultados):,} coincidencias")

print(f"✅ Listo. Líneas leídas: {contador:,} | Registros encontrados: {len(resultados):,}")

## 🧱 Bloque 6 — Construcción del DataFrame de resultados
Se conservan **todas** las coincidencias (si una cuenta aparece varias veces, salen varias filas).

In [ ]:
df_resultados = pd.DataFrame(
    resultados,
    columns=['Cuenta encontrada', 'Registro completo encontrado']
)

# Cuentas que NO se encontraron (útil para revisar después)
no_encontradas = sorted(cuentas_set - cuentas_encontradas)
df_no = pd.DataFrame({'Cuenta no encontrada': no_encontradas})

print(f"Filas en el resultado: {len(df_resultados):,}")
df_resultados.head(10)

## 💾 Bloque 7 — Exportación a Excel y TXT (guardado en Drive)
Crea la carpeta `/Resultados_Busqueda` en tu Drive si no existe y guarda ambos archivos con
fecha y hora en el nombre. El Excel incluye una hoja extra con las cuentas **no encontradas**.

In [ ]:
from datetime import datetime

# 👉 PUEDES MODIFICAR: carpeta de resultados dentro de tu Drive
CARPETA_RESULTADOS = '/content/drive/MyDrive/Resultados_Busqueda'
os.makedirs(CARPETA_RESULTADOS, exist_ok=True)   # la crea si no existe

marca = datetime.now().strftime('%Y%m%d_%H%M%S')
ruta_xlsx = os.path.join(CARPETA_RESULTADOS, f'RESULTADO_BUSQUEDA_{marca}.xlsx')
ruta_otxt = os.path.join(CARPETA_RESULTADOS, f'RESULTADO_BUSQUEDA_{marca}.txt')

# --- Excel ---
LIMITE_EXCEL = 1_048_575  # límite de filas de una hoja de Excel
if len(df_resultados) > LIMITE_EXCEL:
    print("⚠️  Hay más filas que el límite de Excel; también guardaré un CSV completo.")
    df_resultados.to_csv(os.path.join(CARPETA_RESULTADOS,
                         f'RESULTADO_BUSQUEDA_{marca}.csv'), index=False)

with pd.ExcelWriter(ruta_xlsx, engine='openpyxl') as writer:
    df_resultados.head(LIMITE_EXCEL).to_excel(writer, sheet_name='Encontrados', index=False)
    df_no.to_excel(writer, sheet_name='No_encontradas', index=False)

# --- TXT (escrito directo desde la lista: rápido y con poca memoria) ---
with open(ruta_otxt, 'w', encoding='utf-8') as f:
    for _cuenta, linea in resultados:
        f.write(linea + '\n')

print("✅ Archivos guardados:")
print("   📘", ruta_xlsx)
print("   📄", ruta_otxt)

## 📈 Bloque 8 — Resumen final de ejecución

In [ ]:
total_leidas        = len(cuentas_lista)
total_unicas        = len(cuentas_set)
total_encontradas   = len(cuentas_encontradas)
total_no_encontradas = total_unicas - total_encontradas
total_registros     = len(resultados)

print("="*52)
print("           RESUMEN DE LA BÚSQUEDA")
print("="*52)
print(f" Cuentas leídas (filas Excel) : {total_leidas:,}")
print(f" Cuentas únicas a buscar      : {total_unicas:,}")
print(f" Cuentas ENCONTRADAS          : {total_encontradas:,}")
print(f" Cuentas NO encontradas       : {total_no_encontradas:,}")
print(f" Registros (líneas) hallados  : {total_registros:,}")
print("-"*52)
print(f" Carpeta de resultados        : {CARPETA_RESULTADOS}")
print(f" Excel  : {os.path.basename(ruta_xlsx)}")
print(f" TXT    : {os.path.basename(ruta_otxt)}")
print("="*52)